> **Prerequisite:** Run earlier notebooks in numeric order in the **same kernel session**.

# 01 — Statistical models

Imports and regression, correlation, moderation, and moderated mediation.

In [1]:
"""Statistical models for macro-scale gradients figure (HMM states 3-6)."""

from __future__ import annotations



import warnings



import numpy as np

import pandas as pd

from scipy import stats as scipy_stats

from statsmodels.api import OLS, add_constant

from statsmodels.stats.outliers_influence import variance_inflation_factor



warnings.filterwarnings("ignore")

def zscore_series(s: pd.Series) -> pd.Series:
    return (s - s.mean()) / s.std(ddof=1)


def fit_gradient_regression(df: pd.DataFrame) -> dict:
    cols = ["transmodal_distance", "unimodal_distance", "net_SalVentAttn_participation_coefficient_auc"]
    raw = df[cols].dropna().copy()
    sub = raw.copy()
    for c in cols:
        sub[c] = zscore_series(sub[c])
    x = add_constant(sub[["transmodal_distance", "unimodal_distance"]])
    model = OLS(sub["net_SalVentAttn_participation_coefficient_auc"], x).fit()
    vifs = {cols[i]: float(variance_inflation_factor(x.values, i + 1)) for i in range(2)}
    rs = np.random.RandomState(42)
    boot_r2 = []
    n = len(sub)
    arr = sub.values
    for _ in range(2000):
        idx = rs.randint(0, n, n)
        try:
            d = pd.DataFrame(arr[idx], columns=cols)
            xx = add_constant(d[["transmodal_distance", "unimodal_distance"]])
            boot_r2.append(OLS(d["net_SalVentAttn_participation_coefficient_auc"], xx).fit().rsquared)
        except Exception:
            pass
    r2_ci = np.percentile(boot_r2, [2.5, 97.5]) if boot_r2 else [np.nan, np.nan]
    return {"model": model, "raw": raw, "sub_z": sub, "vifs": vifs, "r2_ci": r2_ci, "n": n}


def fit_dispersion_correlation(df: pd.DataFrame) -> dict:
    cols = ["net_SalVentAttn_gradient_dispersion_mean", "rt_duration_diff__long-short"]
    raw = df[cols].dropna().copy()
    sub = raw.copy()
    for c in cols:
        sub[c] = zscore_series(sub[c])
    r, p = scipy_stats.pearsonr(sub.iloc[:, 0], sub.iloc[:, 1])
    return {"raw": raw, "sub_z": sub, "r": float(r), "p": float(p), "n": len(raw)}


def fit_dispersion_moderation(df: pd.DataFrame) -> dict:
    cols = [
        "global_modularity_auc",
        "net_SalVentAttn_gradient_dispersion_mean",
        "rt_transition_diff__flexibility_stability",
    ]
    raw = df[cols].dropna().copy()
    sub = raw.copy()
    for c in cols:
        sub[c] = zscore_series(sub[c])
    x, w, y = (sub[c].values for c in cols)
    design = add_constant(np.column_stack([x, w, x * w]))
    model = OLS(y, design).fit()
    b0, b1, b2, b3 = [float(v) for v in model.params]
    p_b3 = float(model.pvalues[3])
    cov = np.asarray(model.cov_params())
    v11, v13, v33 = cov[1, 1], cov[1, 3], cov[3, 3]
    df_r = model.df_resid

    def simple_slope(mz: float) -> tuple[float, float, float]:
        theta = b1 + b3 * mz
        var = v11 + 2 * mz * v13 + mz**2 * v33
        se = np.sqrt(max(var, 0))
        lo, hi = theta - 1.96 * se, theta + 1.96 * se
        p = 2 * (1 - scipy_stats.t.cdf(abs(theta / se), df_r)) if se > 0 else np.nan
        return theta, p, se

    rs = np.random.RandomState(42)
    boot_r2 = []
    n = len(x)
    for _ in range(2000):
        idx = rs.randint(0, n, n)
        try:
            boot_r2.append(
                OLS(
                    y[idx],
                    add_constant(np.column_stack([x[idx], w[idx], x[idx] * w[idx]])),
                ).fit().rsquared
            )
        except Exception:
            pass
    r2_ci = np.percentile(boot_r2, [2.5, 97.5]) if boot_r2 else [np.nan, np.nan]
    return {
        "model": model,
        "b1": b1,
        "b2": b2,
        "b3": b3,
        "p_b3": p_b3,
        "r2": float(model.rsquared),
        "r2_ci": r2_ci,
        "raw": raw,
        "sub_z": sub,
        "simple_slope": simple_slope,
        "df_resid": df_r,
    }


MODERATED_MEDIATION_COLS = [
    "net_SalVentAttn_participation_coefficient_auc",
    "global_modularity_auc",
    "net_SalVentAttn_gradient_dispersion_mean",
    "rt_transition_diff__flexibility_stability",
]


def fit_moderated_mediation(df: pd.DataFrame) -> dict:
    cols = MODERATED_MEDIATION_COLS
    raw = df[cols].dropna().copy()
    sub = raw.copy()
    for c in cols:
        sub[c] = zscore_series(sub[c])
    x, m, w, y = (sub[c].values for c in cols)
    n = len(x)
    m_m = OLS(m, add_constant(x)).fit()
    a1 = float(m_m.params[1])
    p_a1 = float(m_m.pvalues[1])
    ci_a1 = np.asarray(m_m.conf_int())[1].tolist()
    design_y = np.column_stack([np.ones(n), m, w, m * w, x])
    m_y = OLS(y, design_y).fit()
    b1, b2, b3 = (float(m_y.params[i]) for i in range(1, 4))
    p_b1, p_b3 = float(m_y.pvalues[1]), float(m_y.pvalues[3])
    ci_y = np.asarray(m_y.conf_int())
    ci_b1, ci_b3 = ci_y[1].tolist(), ci_y[3].tolist()
    imm = a1 * b3
    rs = np.random.RandomState(42)
    boot_imm, boot_r2_m, boot_r2_y = [], [], []
    boot_ind_lo, boot_ind_hi = [], []
    for _ in range(5000):
        idx = rs.randint(0, n, n)
        try:
            ab = OLS(m[idx], add_constant(x[idx])).fit().params[1]
            gp = OLS(
                y[idx],
                add_constant(np.column_stack([m[idx], w[idx], m[idx] * w[idx], x[idx]])),
            ).fit()
            boot_imm.append(ab * gp.params[3])
            boot_r2_m.append(OLS(m[idx], add_constant(x[idx])).fit().rsquared)
            boot_r2_y.append(gp.rsquared)
            boot_ind_lo.append(ab * (gp.params[1] + gp.params[3]))
            boot_ind_hi.append(ab * (gp.params[1] - gp.params[3]))
        except Exception:
            pass
    imm_ci = np.percentile(boot_imm, [2.5, 97.5]) if boot_imm else [np.nan, np.nan]
    r2_m_ci = np.percentile(boot_r2_m, [2.5, 97.5]) if boot_r2_m else [np.nan, np.nan]
    r2_y_ci = np.percentile(boot_r2_y, [2.5, 97.5]) if boot_r2_y else [np.nan, np.nan]
    ind_lo = a1 * (b1 + b3)
    ind_hi = a1 * (b1 - b3)
    ind_lo_ci = np.percentile(boot_ind_lo, [2.5, 97.5]) if boot_ind_lo else [np.nan, np.nan]
    ind_hi_ci = np.percentile(boot_ind_hi, [2.5, 97.5]) if boot_ind_hi else [np.nan, np.nan]
    return {
        "n": n,
        "a1": a1, "p_a1": p_a1, "ci_a1": ci_a1,
        "b1": b1, "b3": b3, "p_b1": p_b1, "p_b3": p_b3,
        "ci_b1": ci_b1, "ci_b3": ci_b3,
        "imm": imm, "imm_ci": imm_ci,
        "ind_lo": ind_lo, "ind_hi": ind_hi,
        "ind_lo_ci": ind_lo_ci, "ind_hi_ci": ind_hi_ci,
        "ind_low": ind_lo, "ind_high": ind_hi,
        "ind_low_ci": ind_lo_ci, "ind_high_ci": ind_hi_ci,
        "r2_m": float(m_m.rsquared), "r2_m_ci": r2_m_ci,
        "r2_y": float(m_y.rsquared), "r2_y_ci": r2_y_ci,
        "f_p_m": float(m_m.f_pvalue), "f_p_y": float(m_y.f_pvalue),
        "raw": raw, "sub_z": sub,
    }
